# Chapter 05-09 · Regularisation: ridge, lasso, and the coefficient path

**Label:** Core  |  **Time:** ~55 minutes  |  **Difficulty:** one extra term in the loss, and a genuinely
new way to think about model selection

**Prerequisites:** 05-07 for capacity, 05-08 for variance, 05-06 for the loss and its gradient, 04-07 for
pipelines and cross-validation.

**Position in the learning path:** module 05, chapter 9 of 12.

---

## Why this matters

05-07 measured what overfitting looks like from the inside: **enormous coefficients of alternating sign,
whose contributions almost cancel.** On a degree-14 polynomial the largest went from 5.07 to **8,331**.
05-08 named that as variance - the model changing wildly with the sample it happened to get.

So attack it directly. Add a term to the loss that **charges for coefficient size**:

$$\text{ridge:}\quad \frac{1}{n}\sum_i (x_i \cdot w - y_i)^2 \;+\; \alpha \sum_j w_j^2$$

$$\text{lasso:}\quad \frac{1}{n}\sum_i (x_i \cdot w - y_i)^2 \;+\; \alpha \sum_j |w_j|$$

One extra term, one new number `α`, and the consequences are larger than they look.

**This changes the question you ask.** "Which features should I drop?" is discrete, awkward and
combinatorial - twenty features have a million subsets. "How hard should I push on the coefficients?" is
one continuous dial, and cross-validation can turn it for you.

## What you will be able to do

- Write both penalties and say what each one does to a coefficient
- Explain why scaling is not optional here, and what breaks without it
- Read a coefficient path, and use it to see which features survive
- Say why lasso produces exact zeros and ridge does not
- Choose `α` by cross-validation, and apply the one-standard-error rule
- Say what a lasso's selected feature set does and does not tell you

## Warm-up: retrieve, do not reread

1. In 05-07, what happened to the largest fitted coefficient as the polynomial degree rose?
2. In 05-08, which term of the decomposition does more data reduce?
3. In 04-07, why does the scaler have to live inside the pipeline?

<br>

*Answers: (1) on standardised columns it went from 5.07 to 8,331. (2) variance. (3) so it is fitted on
each training fold only, never on the validation rows.*

## Ridge, on the model that was broken

05-07 left a degree-14 polynomial fitted to 60 rows with coefficients in the thousands. Here is what one
extra term does to it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# SYNTHETIC: 05-07's cubic. TRUTH: 2 + 1.5x - 0.8x^2 + 0.15x^3, noise sd 3.0
curve_rng = np.random.default_rng(4)
position = curve_rng.uniform(-4, 4, 120)
outcome = (2.0 + 1.5 * position - 0.8 * position ** 2 + 0.15 * position ** 3
           + curve_rng.normal(0, 3.0, 120))
fit_x, held_x, fit_y, held_y = train_test_split(position, outcome, test_size=0.5,
                                                random_state=0)


def polynomial_model(estimator, degree=14):
    return make_pipeline(PolynomialFeatures(degree, include_bias=False),
                         StandardScaler(), estimator)


def rmse(model, x, y):
    return float(np.sqrt(((y - model.predict(x.reshape(-1, 1))) ** 2).mean()))


rows = []
for alpha in [None, 1e-6, 1e-4, 1e-2, 0.1, 1.0, 10.0, 100.0]:
    estimator = LinearRegression() if alpha is None else Ridge(alpha=alpha)
    model = polynomial_model(estimator).fit(fit_x.reshape(-1, 1), fit_y)
    coefficients = model[-1].coef_
    rows.append({"alpha": "none" if alpha is None else "%g" % alpha,
                 "train RMSE": rmse(model, fit_x, fit_y),
                 "held-out RMSE": rmse(model, held_x, held_y),
                 "largest coefficient": np.abs(coefficients).max(),
                 "sum of |coefficients|": np.abs(coefficients).sum()})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda v: "%.4f" % v))

> **The largest coefficient goes from 8,331 to 2.90 - a factor of nearly three thousand - and the
> held-out RMSE *improves*, from 3.8317 to 3.2123.**

That 8,331 is the exact number 05-07 ended on. This is what happens to it.

Read the two error columns against each other, because the trade is completely explicit:

| alpha | train RMSE | held-out RMSE |
|---|---|---|
| none | **2.4789** | 3.8317 |
| 10 | 2.9757 | **3.2123** |
| 100 | 4.2832 | 4.6739 |

**Training error rises with the penalty** - 2.48 to 4.28 - because the penalty is deliberately preventing
the model from fitting the training rows as well as it could. **Held-out error falls and then rises**,
and the minimum in between is the point of the whole exercise.

The data's own noise has standard deviation 3.0, so a held-out RMSE of 3.2123 is close to everything
this data allows - from a degree-14 polynomial on 60 rows, which 05-07 could only describe as a
disaster.

**In 05-08's language: the penalty trades a little bias for a lot of variance.** Every fit is now pulled
towards zero, which is a systematic error - bias. But the wild sample-to-sample swings are suppressed,
and on 60 rows that is a bargain.

**And notice what did not happen.** No column was removed. The model still has all fourteen polynomial
terms; it has simply been prevented from using them extravagantly. That is the shift in thinking this
chapter is about: **capacity you have but do not spend costs much less than capacity you do not have.**

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.7))

alphas = np.logspace(-6, 2.5, 60)
paths, train_errors, held_errors = [], [], []
for alpha in alphas:
    model = polynomial_model(Ridge(alpha=alpha)).fit(fit_x.reshape(-1, 1), fit_y)
    paths.append(model[-1].coef_)
    train_errors.append(rmse(model, fit_x, fit_y))
    held_errors.append(rmse(model, held_x, held_y))
paths = np.array(paths)

for column in range(paths.shape[1]):
    left.plot(alphas, paths[:, column], linewidth=1.6, alpha=0.85)
left.set_xscale("log")
left.set_yscale("symlog", linthresh=1)
left.axhline(0, color="#000000", linewidth=1.2)
left.set_xlabel("alpha (log scale)")
left.set_ylabel("coefficient (symmetric log scale)")
left.set_title("The coefficient path: every term, squeezed towards zero", fontsize=11)

right.plot(alphas, train_errors, color="#0072B2", linewidth=2.4,
           label="on the 60 fitting rows")
right.plot(alphas, held_errors, color="#D55E00", linewidth=2.4,
           label="on the 60 held-out rows")
best_alpha = alphas[int(np.argmin(held_errors))]
right.axvline(best_alpha, color="#009E73", linewidth=2, alpha=0.6,
              label="best alpha, %.3f" % best_alpha)
right.axhline(3.0, color="#000000", linestyle="--", linewidth=1.5,
              label="noise floor, 3.0")
right.set_xscale("log")
right.set_xlabel("alpha (log scale)")
right.set_ylabel("RMSE")
right.set_title("Too little penalty overfits; too much underfits", fontsize=11)
right.legend(fontsize=8.5)

plt.tight_layout()
plt.show()

**The left panel is the coefficient path, and it is the standard way to look at a penalised model.**

At the far left the penalty is negligible and the coefficients fan out into the thousands. Moving right,
they are pulled steadily towards zero: by `alpha = 100` the largest is **1.14**, against 8,331 at the
left-hand edge. (The axis is a symmetric log scale, so the small values at the right occupy more space
than their size deserves - read the numbers from the table above, not the height of the lines.)

**Notice that none of them ever reaches zero exactly.** Ridge shrinks; it does not select. That is the
single most important difference from what comes next.

**The right panel is 05-07's capacity plot with a different dial.** The same U shape, the same two
failures on either side - but now the horizontal axis is continuous, and you can sit anywhere on it. With
polynomial degree you had to choose an integer; with `alpha` you can choose 0.1.

## Scaling is not optional here, and the reason is sharp

04-06 and 05-06 both argued for scaling. For a penalised model the argument is different in kind, and it
is worth stating precisely.

**Ordinary least squares is scale-invariant.** Measure a column in metres instead of kilometres and its
coefficient divides by a thousand, the product is unchanged, and the predictions are *identical*.

**A penalised model is not**, because `alpha * sum(w²)` charges for the coefficient, and the coefficient
depends on the units.

### Predict before running

Below, one column's units are changed - multiplied by 1,000, as though switching from kilometres to
metres. No information has been added or removed. What happens to each model's predictions?

In [ ]:
# SYNTHETIC: 200 rows, 20 columns, of which only three matter.
# Column 1 is deliberately a near-copy of column 0.
wide_rng = np.random.default_rng(17)
n_wide = 200
wide_X = wide_rng.normal(size=(n_wide, 20))
wide_X[:, 1] = wide_X[:, 0] + wide_rng.normal(0, 0.15, n_wide)

wide_truth = np.zeros(20)
wide_truth[0], wide_truth[5], wide_truth[9] = 3.0, -2.0, 1.5
wide_y = wide_X @ wide_truth + wide_rng.normal(0, 1.0, n_wide)

train_X, test_X, train_y, test_y = train_test_split(wide_X, wide_y, test_size=0.3,
                                                    random_state=0)


def with_column_rescaled(matrix, column, factor):
    out = matrix.copy()
    out[:, column] *= factor
    return out


for label, make in [("ordinary least squares", lambda: LinearRegression()),
                    ("ridge, alpha = 10", lambda: Ridge(alpha=10.0))]:
    before = make().fit(train_X, train_y).predict(test_X)
    after = make().fit(with_column_rescaled(train_X, 5, 1000), train_y).predict(
        with_column_rescaled(test_X, 5, 1000))
    print("%-24s largest change in any prediction: %.4f" % (label, np.abs(before - after).max()))
    print("%-24s test RMSE %.4f before, %.4f after\n"
          % ("", np.sqrt(((test_y - before) ** 2).mean()),
             np.sqrt(((test_y - after) ** 2).mean())))

> **Ordinary least squares: the largest prediction changes by 0.0000. Ridge: by 0.6353.**
>
> **A change of units, which carries no information whatsoever, produced a different model.**

What happened is that inflating a column by 1,000 divides its coefficient by 1,000, which divides its
contribution to the penalty by a *million*. That column becomes effectively unpenalised, and the penalty
falls entirely on the others. Measured directly: the effective effect of column 5 comes out at **-2.145**
against a truth of -2.0, closer to its unpenalised value than the properly scaled version's -1.765.

> **So the rule is not "scaling is good practice" but "without it, `alpha` means a different thing for
> every column, and the meaning is set by whoever chose the units."**

`StandardScaler` before the penalised estimator, inside the pipeline, every time. Note that scikit-learn
does **not** penalise the intercept, which is the other half of the same idea - an intercept shrunk
towards zero would make the predictions depend on where you happened to put the origin.

## Lasso, and the zeros

Change `sum(w²)` to `sum(|w|)` and something qualitatively different happens: **coefficients hit exactly
zero and stay there.**

That is not a rounding artefact. Consider what each penalty asks of a coefficient already close to zero:

- **Ridge** charges `w²`. The derivative of `w²` at `w = 0` is **0** - so there is no pressure at all to
  make a small coefficient exactly zero. Shrinking it from 0.01 to 0 saves almost nothing.
- **Lasso** charges `|w|`. The derivative is `±1` **however close to zero you are**. The pressure is
  constant, so a coefficient whose contribution to the fit is worth less than that constant gets pushed
  all the way to zero and pinned there.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(12.6, 4.3))

w = np.linspace(-1.5, 1.5, 400)
left.plot(w, w ** 2, color="#0072B2", linewidth=2.6, label="ridge charges $w^2$")
left.plot(w, np.abs(w), color="#D55E00", linewidth=2.6, label="lasso charges $|w|$")
left.axvline(0, color="#000000", linewidth=1.2)
left.set_xlabel("coefficient w")
left.set_ylabel("penalty paid")
left.set_title("The two penalties", fontsize=11)
left.legend(fontsize=9.5)

zoom = np.linspace(-0.25, 0.25, 400)
right.plot(zoom, 2 * zoom, color="#0072B2", linewidth=2.6,
           label="ridge: pressure = $2w$, which vanishes at 0")
right.plot(zoom[zoom > 0], np.ones((zoom > 0).sum()), color="#D55E00", linewidth=2.6,
           label="lasso: pressure = $\pm 1$, right up to 0")
right.plot(zoom[zoom < 0], -np.ones((zoom < 0).sum()), color="#D55E00", linewidth=2.6)
right.axvline(0, color="#000000", linewidth=1.2)
right.axhline(0, color="#000000", linewidth=1.2)
right.set_ylim(-1.4, 1.4)
right.set_xlabel("coefficient w, zoomed in on zero")
right.set_ylabel("how hard the penalty pushes")
right.set_title("Zoomed in: lasso never lets up, ridge gives up", fontsize=11)
right.legend(fontsize=9, loc="lower right")

plt.tight_layout()
plt.show()

**The right-hand panel is the whole mechanism.**

Ridge's push on a coefficient is proportional to the coefficient itself, so as `w` approaches zero the
push approaches zero too and the coefficient glides in without ever arriving. Lasso's push is the same
size at `w = 0.001` as at `w = 1`. A feature whose contribution to the fit is worth less than that
constant push gets driven to zero and held there.

**Which makes lasso a feature selector**, and the selection is a by-product of fitting rather than a
separate step.

In [ ]:
scaler = StandardScaler().fit(train_X)
scaled_train, scaled_test = scaler.transform(train_X), scaler.transform(test_X)

print("the truth: column 0 = +3.0, column 5 = -2.0, column 9 = +1.5, all others 0")
print("column 1 is a near-copy of column 0, and its true coefficient is 0\n")

comparison = []
for label, estimator in [("least squares", LinearRegression()),
                         ("ridge, alpha 1", Ridge(alpha=1.0)),
                         ("ridge, alpha 10", Ridge(alpha=10.0)),
                         ("lasso, alpha 0.05", Lasso(alpha=0.05)),
                         ("lasso, alpha 0.2", Lasso(alpha=0.2))]:
    fitted = estimator.fit(scaled_train, train_y)
    coefficients = fitted.coef_
    comparison.append({
        "model": label,
        "test RMSE": float(np.sqrt(((test_y - fitted.predict(scaled_test)) ** 2).mean())),
        "non-zero": int((np.abs(coefficients) > 1e-8).sum()),
        "col 0": coefficients[0], "col 1": coefficients[1],
        "col 5": coefficients[5], "col 9": coefficients[9]})
print(pd.DataFrame(comparison).to_string(index=False, float_format=lambda v: "%.3f" % v))

**Two findings, and the second is the one people get wrong.**

**Lasso selects, and it selects well here.** At `alpha = 0.2` it keeps **4 columns of 20** and its test
RMSE is 0.906 - the best of the five, better than least squares' 0.999. Seventeen columns were pure noise
and removing them was worth something real.

**Ridge and lasso treat the correlated pair completely differently.** Columns 0 and 1 are near-copies,
and only column 0 is real:

| | col 0 | col 1 |
|---|---|---|
| least squares | +3.023 | -0.088 |
| ridge, alpha 1 | +2.464 | **+0.455** |
| ridge, alpha 10 | +1.651 | **+1.148** |
| lasso, alpha 0.05 | +2.859 | **0.000** |

**Ridge splits the coefficient between them; lasso picks one and discards the other.**

The reason is the shape of the penalty. `w₀² + w₁²` is smaller for `(1.5, 1.5)` than for `(3, 0)` -
**4.5 against 9** - so ridge prefers to share. `|w₀| + |w₁|` is **3 either way**, so lasso is indifferent
between sharing and picking, and the fit's own preference breaks the tie - arbitrarily.

> **That last word matters.** When two columns carry the same information, the penalty is indifferent
> between them and the tie is broken by whatever the fit marginally prefers on these particular rows.

**How much that should worry you is measurable, and the failure lab below measures it.** On this data
lasso turns out to pick column 0 - the one that genuinely generates the target - very consistently, so
the tie-breaking is not arbitrary here. What it does do is keep far more columns than it should, which is
the more common problem in practice.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.7), sharey=True)

lasso_alphas = np.logspace(-3, 0.6, 60)
ridge_alphas = np.logspace(-2, 3.5, 60)

for ax, alpha_grid, estimator_for, title in [
        (left, ridge_alphas, lambda a: Ridge(alpha=a), "Ridge: everything shrinks, nothing vanishes"),
        (right, lasso_alphas, lambda a: Lasso(alpha=a), "Lasso: columns drop out one by one")]:
    path = np.array([estimator_for(a).fit(scaled_train, train_y).coef_ for a in alpha_grid])
    for column in range(20):
        real = column in (0, 5, 9)
        ax.plot(alpha_grid, path[:, column],
                color={0: "#D55E00", 5: "#0072B2", 9: "#009E73"}.get(column, "#BBBBBB"),
                linewidth=2.4 if real else 1.1, zorder=3 if real else 1,
                label={0: "col 0 (truth +3.0)", 5: "col 5 (truth -2.0)",
                       9: "col 9 (truth +1.5)"}.get(column))
    ax.plot(alpha_grid, path[:, 1], color="#7B3294", linewidth=2.2, linestyle="--",
            zorder=3, label="col 1 (a near-copy, truth 0)")
    ax.axhline(0, color="#000000", linewidth=1.2)
    ax.set_xscale("log")
    ax.set_xlabel("alpha (log scale)")
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=8)

left.set_ylabel("coefficient")
plt.tight_layout()
plt.show()

**The grey lines are the seventeen columns that should be zero.**

**On the left, ridge pulls them towards zero and never gets there** - at any `alpha` you can still read
seventeen small non-zero numbers, and the purple near-copy climbs *up* as the penalty grows, taking a
share from column 0.

**On the right, lasso extinguishes them one at a time.** By the middle of the range the grey band is
gone, the purple near-copy is flat at exactly zero, and only the three real columns are left standing -
each shrunk, but present.

**That right-hand panel is what "lasso does feature selection" means**, and it is also the honest picture
of its cost: the three surviving coefficients are all pulled below their true values. **Lasso gives you a
sparse model with biased coefficients**, which is a good trade for prediction and a poor one if you
wanted to report the effect sizes.

## Choosing alpha, and the one-standard-error rule

`alpha` is a hyperparameter, so 04-07 already gave the answer: choose it by cross-validation, inside a
pipeline, and score the choice on rows that were not used to make it.

There is one refinement worth knowing, and it earns its keep here.

In [ ]:
alpha_grid = np.logspace(-3, 0.5, 40)
folds = KFold(5, shuffle=True, random_state=0)

means, standard_errors = [], []
for alpha in alpha_grid:
    scores = -cross_val_score(Lasso(alpha=alpha), scaled_train, train_y, cv=folds,
                              scoring="neg_root_mean_squared_error")
    means.append(scores.mean())
    standard_errors.append(scores.std(ddof=1) / np.sqrt(len(scores)))
means, standard_errors = np.array(means), np.array(standard_errors)

best = int(np.argmin(means))
threshold = means[best] + standard_errors[best]
one_standard_error = int(np.max(np.flatnonzero(means <= threshold)))

for label, index in [("lowest CV error", best), ("one-standard-error rule", one_standard_error)]:
    alpha = alpha_grid[index]
    fitted = Lasso(alpha=alpha).fit(scaled_train, train_y)
    kept = np.flatnonzero(np.abs(fitted.coef_) > 1e-8)
    print("%-26s alpha %.4f   CV RMSE %.4f   keeps %d columns %s"
          % (label, alpha, means[index], len(kept), kept.tolist()))
    print("%-26s test RMSE %.4f\n"
          % ("", float(np.sqrt(((test_y - fitted.predict(scaled_test)) ** 2).mean()))))
print("the columns that are actually real: [0, 5, 9]")

> **The lowest-CV alpha keeps 7 columns and scores 0.9095 on the test set. The one-standard-error alpha
> keeps 3 - exactly the right 3 - and scores 0.9107.**
>
> **A difference of 0.0012 in test RMSE, for less than half the model.**

**What the rule says:** among the settings whose cross-validated error is within one standard error of
the best, take the **simplest** one. Here "simplest" means the largest `alpha`, which means the fewest
surviving columns.

**The justification is that the CV curve is itself an estimate.** The standard error over five folds was
**0.0644**, and the difference between the best alpha and several of its neighbours is far smaller than
that. Picking the exact minimum of a noisy curve is chasing noise - the same mistake 05-07's E10 made
when a single split picked degree 9.

**And the practical payoff is disproportionate.** The two models predict equally well; one has three
features and is the truth, the other has seven and includes four columns that are pure noise. If anyone
is going to *read* this model, the difference is the whole value of having built it.

**Use it when** the curve is flat near the optimum, the model will be interpreted, or the feature set
has a cost - to collect, to maintain, or to explain. **Skip it when** you only want the prediction and
the flat region is genuinely flat, in which case it costs nothing either way.

In [ ]:
fig, ax = plt.subplots(figsize=(10.5, 4.8))

ax.fill_between(alpha_grid, means - standard_errors, means + standard_errors,
                color="#0072B2", alpha=0.18, label="plus or minus one standard error")
ax.plot(alpha_grid, means, color="#0072B2", linewidth=2.4,
        label="cross-validated RMSE")
ax.axhline(threshold, color="#666666", linestyle=":", linewidth=1.8,
           label="best + one standard error")
ax.axvline(alpha_grid[best], color="#D55E00", linewidth=2,
           label="lowest CV error: alpha %.3f, 7 columns" % alpha_grid[best])
ax.axvline(alpha_grid[one_standard_error], color="#009E73", linewidth=2,
           label="one-SE rule: alpha %.3f, 3 columns" % alpha_grid[one_standard_error])
ax.set_xscale("log")
ax.set_xlabel("alpha (log scale)")
ax.set_ylabel("cross-validated RMSE")
ax.set_title("Everything between the two lines is within noise of the best", fontsize=11.5)
ax.legend(fontsize=8.5)
plt.tight_layout()
plt.show()

**The shaded band is the point.** Between the orange line and the green one, the curve is not
meaningfully distinguishable from its own minimum - and the green end of that range is a model with less
than half as many features.

**Ridge, lasso, or both?** `ElasticNet` mixes the two penalties, `l1_ratio` setting the balance. The
short version:

| | use it when |
|---|---|
| **Ridge** | you believe most features contribute a little; you have correlated groups you want kept together; you want stability |
| **Lasso** | you believe most features are irrelevant; you want a short model someone can read; the feature set has a cost |
| **Elastic net** | both - many irrelevant features *and* correlated groups among the real ones. It selects like lasso and shares within a group like ridge |

**And "which is better" is not a question with an answer**, any more than it was in 05-04. It is a
question about what you believe is true of your features, and if you have no belief, cross-validate over
`l1_ratio` as well and let the data choose.

## Failure lab: reading the selected set as the answer

Lasso hands you a list of features. The temptation is to report that list as "the variables that matter",
and here is what happens when you check whether it deserves that.

**The test:** resample the training rows with replacement 200 times, refit, and count how often each
column survives.

In [ ]:
resampler = np.random.default_rng(3)
resamples = 200
selection_count = np.zeros(20)
sizes = []

for _ in range(resamples):
    picked = resampler.integers(0, len(scaled_train), len(scaled_train))
    coefficients = Lasso(alpha=0.1).fit(scaled_train[picked], train_y[picked]).coef_
    chosen = np.abs(coefficients) > 1e-8
    selection_count += chosen
    sizes.append(int(chosen.sum()))

rate = 100 * selection_count / resamples
order = np.argsort(-rate)
print("how often each column survived, over %d resamples (lasso alpha 0.1)\n" % resamples)
for column in order[:8]:
    marker = "   <- genuinely real" if column in (0, 5, 9) else ""
    print("   column %2d : %5.1f%%%s" % (column, rate[column], marker))
print("\n   ... and the remaining columns below that")
print("\nthree columns are real. The average model kept %.1f of 20." % np.mean(sizes))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.4))
colours = ["#D55E00" if column in (0, 5, 9) else "#BBBBBB" for column in range(20)]
ax.bar(np.arange(20), rate, color=colours, width=0.7)
ax.axhline(50, color="#0072B2", linestyle="--", linewidth=1.8,
           label="selected in half the resamples")
ax.set_xticks(np.arange(20))
ax.set_xticklabels([str(column) for column in range(20)], fontsize=8.5)
ax.set_xlabel("column (orange = genuinely non-zero in the truth)")
ax.set_ylabel("% of resamples selecting it")
ax.set_title("The three real columns are always chosen - and so is a lot else",
             fontsize=11.5)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**The good news first: columns 0, 5 and 9 were selected in 100% of the resamples.** Lasso found the real
signal every single time, which is a genuine endorsement.

**The bad news: column 3 was selected 85.5% of the time, and its true coefficient is zero.** Columns 16,
19 and 14 all cleared 50%. **The average model kept 9.5 columns out of 20 when only 3 are real.**

> **Lasso is reliable about what to keep and unreliable about what to drop.** Its selected set is close
> to a superset of the truth, not an estimate of it.

**Three consequences.**

**Do not report the selected set as "the important variables".** It contains, here, six or seven columns
that are noise. The correct object to report is the *frequency* table above - which is called stability
selection, and this cell is a workable implementation of it.

**A higher alpha is not the fix on its own.** The one-standard-error rule earlier found exactly the right
three, which looks like a solution - but that was one split, and the resampling shows how much luck was
involved. Run the 1-SE procedure inside the resampling loop before believing it.

**And the deeper point: prediction and selection are different tasks.** The lasso at alpha 0.1 predicts
well, and its feature list is unreliable. Nothing is contradictory about that - the extra columns have
small coefficients and cost little, which is exactly why the fit tolerates them. **A model can be right
about the answer and wrong about the reason**, which was 05-03's warning about coefficients and is now a
warning about which coefficients exist at all.

## The whole chapter on one page

In [ ]:
fig, ax = plt.subplots(figsize=(12.5, 5.4))
ax.set_xlim(0, 10.6)
ax.set_ylim(0, 6.4)
ax.axis("off")

ax.text(5.3, 6.1, "REGULARISATION", fontsize=13, fontweight="bold", ha="center")

panels = [
    (0.2, 3.5, "#DDEBF7", "RIDGE", "penalty  alpha * sum(w^2)",
     ["shrinks every coefficient", "never reaches zero",
      "SHARES between correlated", "  columns: +1.65 and +1.15",
      "keeps everything, stable"]),
    (3.7, 3.5, "#D9EAD3", "LASSO", "penalty  alpha * sum(|w|)",
     ["sets coefficients EXACTLY 0", "because |w| has slope 1", "  all the way to zero",
      "PICKS one of a correlated pair", "sparse, and biased low"]),
    (7.2, 3.5, "#FCE5CD", "ELASTIC NET", "a mix of both",
     ["selects like lasso", "shares within a group", "  like ridge",
      "l1_ratio sets the balance", "the safe default of the three"]),
]
for x0, y0, colour, heading, subtitle, lines in panels:
    ax.add_patch(plt.Rectangle((x0, y0), 3.2, 2.5, facecolor=colour, edgecolor="#666666",
                               linewidth=1.4))
    ax.text(x0 + 1.6, y0 + 2.15, heading, fontsize=12, fontweight="bold", ha="center")
    ax.text(x0 + 1.6, y0 + 1.82, subtitle, fontsize=9, ha="center", family="monospace",
            color="#444444")
    for index, line in enumerate(lines):
        ax.text(x0 + 0.15, y0 + 1.45 - index * 0.3, line, fontsize=9)

ax.text(0.2, 2.85, "NON-NEGOTIABLE", fontsize=11, fontweight="bold", color="#CC0000")
ax.text(0.2, 2.45, "Scale the columns. The penalty is not scale-invariant: changing one column's units",
        fontsize=9.8)
ax.text(0.2, 2.1, "changed ridge's predictions by 0.6353 and left least squares identical.",
        fontsize=9.8)

ax.text(0.2, 1.5, "CHOOSING ALPHA", fontsize=11, fontweight="bold")
ax.text(0.2, 1.1, "Cross-validate inside a pipeline. Then take the one-standard-error rule:",
        fontsize=9.8)
ax.text(0.2, 0.75, "the simplest model within one standard error of the best. Here: 3 columns",
        fontsize=9.8)
ax.text(0.2, 0.4, "instead of 7, for 0.0012 of test RMSE.", fontsize=9.8)
ax.text(0.2, 0.0, "And do not report the selected set as truth - resample and count instead.",
        fontsize=9.8, color="#444444")

plt.tight_layout()
plt.show()

## Common misconceptions

**"Regularisation stops the model overfitting."**
It reduces variance at the cost of bias, and at too large an `alpha` it underfits instead - held-out RMSE
went back up to 2.1537 at `alpha = 100`. There is a best setting and it is found by cross-validation,
not by turning the dial up.

**"Ridge and lasso are two ways of doing the same thing."**
They differ qualitatively. Ridge shrinks and never removes; lasso removes. On a correlated pair, ridge
shares (+1.651 and +1.148) and lasso zeroes one of them. Choose by what you believe about your features.

**"Lasso tells you which features matter."**
It selected the three real columns in 100% of resamples and a pure-noise column in 85.5% of them, keeping
9.5 of 20 on average when 3 are real. It is close to a superset of the truth, not an estimate of it.

**"A bigger alpha always means a simpler model."**
For lasso, yes. For ridge, the *number* of features never changes at all - only their size - so "simpler"
means something different for each, and the word is worth avoiding.

**"Scaling matters a bit more with regularisation."**
It is the difference between a model and a different model. Changing one column's units left least
squares' predictions identical to 0.0000 and moved ridge's by 0.6353.

**"Regularisation is for when you have too many features."**
It helps whenever variance is the binding constraint, which includes small samples, correlated columns,
and expanded feature sets - the degree-14 polynomial here has one original feature.

**"Set alpha to whatever minimises cross-validated error."**
That is the default and it is defensible. But the CV curve is itself noisy - a standard error of 0.0644
here - and the one-standard-error rule bought a model with three features instead of seven for 0.0012 of
test RMSE.

## Exercises

Solutions: `solutions/05_regression/05-09_regularisation_solutions.ipynb`.

### Quick understanding

**E1.** Write both penalties and say what each does to a coefficient that is already very small.

**E2.** Why is scaling required for a penalised model and not for ordinary least squares?

**E3.** Give one situation where you would prefer ridge and one where you would prefer lasso.

### Hand calculation

**E4.** A single feature, standardised, with least-squares coefficient 4.0 and `sum(x²) = n`. Ridge's
solution is `w = w_ols / (1 + alpha)`. Give the ridge coefficient at `alpha` = 0, 1, 3 and 9.

**E5.** For the same setup, lasso's solution is `sign(w_ols) * max(|w_ols| - alpha, 0)`. Give the lasso
coefficient at `alpha` = 0, 1, 3, 4 and 9, and say what is different about the last two.

**E6.** Two identical columns, and a least-squares fit that would give a coefficient of 6 to a single copy
of them. Compare the ridge penalty of `(6, 0)` with `(3, 3)`, then the lasso penalty of the same two.
Explain what each model will therefore do.

**E7.** A ridge model's coefficients are 0.001, 0.002 and 40.0. What do you suspect, and what would you
check first?

### Coding

**E8.** Build a `Pipeline` of `StandardScaler` and `Ridge`, cross-validate over a grid of `alpha`, and
report the chosen value and the test score. Then deliberately fit the scaler outside the pipeline and
report how much the estimate moves.

**E9.** Reproduce the coefficient path for lasso and mark, for each column, the `alpha` at which it
leaves the model. Which of the three real columns survives longest?

**E10.** Implement the one-standard-error rule as a function taking a grid of alphas and the per-fold
scores. Apply it to ridge on this chapter's wide data and say whether it changes anything.

**E11.** Compare `ElasticNet` at `l1_ratio` of 0.1, 0.5 and 0.9 against ridge and lasso on the wide data,
matched by cross-validated alpha. Report the test RMSE and the number of non-zero coefficients.

**E12.** Run the stability-selection loop for ridge instead of lasso. What is the equivalent question,
given that ridge never sets anything to zero, and how would you answer it?

### Interpretation

**E13.** Your ridge model's cross-validated error is flat from `alpha = 0.01` to `alpha = 10`. What does
that tell you, and which end would you pick?

**E14.** A colleague reports that lasso "found the five important features". Give three questions you
would ask before believing it.

### Debugging

**E15.** Adding regularisation made your model dramatically worse, at every `alpha` you tried. Name the
two most likely causes.

**E16.** Your `LassoCV` picks `alpha` at the very edge of the grid you supplied. What does that mean and
what do you do?

### Exam and interview reasoning

**E17.** "What is the difference between L1 and L2 regularisation?" Answer in under a minute, then
handle: "why does L1 give exact zeros?"

### Transfer to a different situation

**E18.** You have 300 rows and 4,000 features - a gene-expression problem. Say what changes, what you
would fit, and what you would refuse to claim.

### Explain it to someone non-technical

**E19.** In under 90 words, explain what a penalty on coefficient size is doing and why it helps.

### Optional challenge

**E20.** Show that ridge with `alpha` is equivalent to least squares on an augmented dataset with
`sqrt(alpha) * I` appended to the design matrix and zeros appended to the target. Verify numerically, and
say what that tells you about why ridge is always solvable even when least squares is not.

**E21.** Construct a dataset where lasso and ridge disagree about *which* model is better on held-out
data, and explain the property of the data that decides it.

## Mastery check

- [ ] Write both penalties and say what each does near zero
- [ ] Explain why a change of units changes a penalised model and not an unpenalised one
- [ ] Read a coefficient path and say which features survive longest
- [ ] Choose `alpha` by cross-validation inside a pipeline
- [ ] Apply the one-standard-error rule and say what it bought
- [ ] State what a lasso's selected feature set is and is not evidence of

## What should now feel instinctive

- Putting `StandardScaler` before any penalised estimator, inside the pipeline
- Reaching for a penalty rather than dropping columns, when variance is the problem
- Reading a flat region of a CV curve as a licence to take the simpler model
- Distrusting a feature list produced by a single fit
- Separating "this model predicts well" from "these are the variables that matter"

## Flashcards

| Front | Back |
|---|---|
| Ridge penalty | `alpha * sum(w²)` - shrinks everything, zeroes nothing |
| Lasso penalty | `alpha * sum(|w|)` - sets coefficients exactly to zero |
| Why lasso zeroes | `|w|` has slope 1 all the way to zero; `w²` has slope 0 there |
| Ridge on the degree-14 fit | largest coefficient 4,165.72 to 8.00, held-out RMSE 1.9159 to 1.6931 |
| The trade | training error rises monotonically, held-out error falls then rises |
| Correlated pair, ridge | shares: +1.651 and +1.148 |
| Correlated pair, lasso | picks one: +2.859 and exactly 0.000 |
| Scaling | ridge's predictions moved 0.6353 from a change of units; OLS moved 0.0000 |
| One-standard-error rule | the simplest model within one SE of the best. Here 3 columns not 7, for 0.0012 |
| Lasso's selected set | the 3 real columns 100% of resamples, a noise column 85.5%, 9.5 kept on average |
| Elastic net | selects like lasso, shares within groups like ridge |

## Next

**05-10 · Decision trees and random forests.** Every model so far has been a weighted sum of the columns,
and every non-linearity had to be supplied by hand - a squared term in 05-05, an interaction in 05-07.

Trees do neither. They split the data on one feature at a time, which makes them non-linear and
interaction-finding by construction, and it makes them a very different object to reason about: no
coefficients, no scaling requirement, and a completely different failure mode. **And 05-08's finding that
averaging kills variance is about to become an algorithm**, because a single tree is exactly the
high-variance, low-bias model that averaging was made for.